<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Reset Keras/TensorFlow session (clean memory)

In [2]:
import tensorflow as tf
tf.keras.backend.clear_session()

Mount Google Drive to access project files

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import required libraries

In [4]:
import tensorflow as tf
import numpy as np
import os
from sklearn.utils import class_weight
import shutil
import csv
from datetime import datetime
import pandas as pd

Define training and testing dataset locations

In [5]:
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"

Load training and testing datasets from Drive into TensorFlow format

In [6]:
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)

Found 1032 files belonging to 8 classes.
Found 264 files belonging to 8 classes.


Retrieve and display class names from dataset

In [7]:
class_names = train_data.class_names
print(class_names)

['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']


Dataset preprocessing (normalize + convert grayscale to RGB)

In [8]:
def preprocess(image, label):
    image = image / 255.0                 # normalize
    image = tf.repeat(image, 3, axis=-1)  # gray → RGB
    return image, label

train_data = train_data.map(preprocess)
test_data  = test_data.map(preprocess)

Verify preprocessed image shape (sanity check)

In [9]:
for images, labels in train_data.take(1):
    print(images.shape)

(32, 224, 224, 3)


Loading a Pretrained Base Model (Feature Extractor)

In [10]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Freezing the pretrained model (Feature Extraction mode)

In [11]:
base_model.trainable = False

Building and Compiling the Final Classification Model

In [12]:
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,016,312 (3.88 MB)

 Trainable params: 76,040 (297.03 KB)

 Non-trainable params: 940,272 (3.59 MB)

Computing class weights to handle imbalanced data

In [13]:
labels = np.concatenate([y for x,y in train_data], axis=0)

cw = class_weight.compute_class_weight(
    "balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(cw))
print("Class weights:", class_weights)

Class weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0), 6: np.float64(1.0), 7: np.float64(1.0)}


Training the model on the dataset

In [14]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=12
)

Epoch 1/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.1756 - loss: 2.1382 - val_accuracy: 0.1780 - val_loss: 2.1014
Epoch 2/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 715ms/step - accuracy: 0.3196 - loss: 1.8358 - val_accuracy: 0.1856 - val_loss: 2.0733
Epoch 3/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 38s 629ms/step - accuracy: 0.3812 - loss: 1.6621 - val_accuracy: 0.1856 - val_loss: 2.0442
Epoch 4/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 43s 679ms/step - accuracy: 0.4579 - loss: 1.5514 - val_accuracy: 0.1894 - val_loss: 2.0110
Epoch 5/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.4963 - loss: 1.4792 - val_accuracy: 0.1970 - val_loss: 1.9756
Epoch 6/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 644ms/step - accuracy: 0.5226 - loss: 1.4225 - val_accuracy: 0.2803 - val_loss: 1.9358
Epoch 7/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 27s 835ms/step - accuracy: 0.5223 - loss: 1.3826 - val_accuracy: 0.3939 - val_loss: 1.8914
Epoch 8/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 36s 682ms/step - accuracy: 0.5287 - loss: 1.3204 - val_accurac

Fine-tuning / Refining the model with smaller learning rate + Early Stopping (Second Training Phase)

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-6),   # very gentle
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_more = model.fit(
    train_data,
    validation_data=test_data,
    epochs=6,
    callbacks=[early]
)

Epoch 1/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 34s 765ms/step - accuracy: 0.6410 - loss: 1.1034 - val_accuracy: 0.6667 - val_loss: 1.4895
Epoch 2/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 37s 648ms/step - accuracy: 0.6633 - loss: 1.1029 - val_accuracy: 0.6894 - val_loss: 1.4281
Epoch 3/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 41s 641ms/step - accuracy: 0.6801 - loss: 1.0580 - val_accuracy: 0.7083 - val_loss: 1.3680
Epoch 4/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 24s 729ms/step - accuracy: 0.6322 - loss: 1.0972 - val_accuracy: 0.7159 - val_loss: 1.3102
Epoch 5/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 648ms/step - accuracy: 0.6282 - loss: 1.1094 - val_accuracy: 0.7197 - val_loss: 1.2556
Epoch 6/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 680ms/step - accuracy: 0.6073 - loss: 1.1142 - val_accuracy: 0.7159 - val_loss: 1.2054


Final fine-tuning with ultra-small learning rate (Third training phase)

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_more = model.fit(
    train_data,
    validation_data=test_data,
    epochs=4,
    callbacks=[early]
)

Epoch 1/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 32s 750ms/step - accuracy: 0.6280 - loss: 1.0914 - val_accuracy: 0.7197 - val_loss: 1.1607
Epoch 2/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 709ms/step - accuracy: 0.6451 - loss: 1.1074 - val_accuracy: 0.7311 - val_loss: 1.1216
Epoch 3/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 639ms/step - accuracy: 0.6308 - loss: 1.1004 - val_accuracy: 0.7348 - val_loss: 1.0884
Epoch 4/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 42s 662ms/step - accuracy: 0.6383 - loss: 1.1342 - val_accuracy: 0.7386 - val_loss: 1.0610


Saving the trained model to Google Drive

In [17]:
model.save("/content/drive/MyDrive/wafer_xai3_model.keras")
print("Saved XAI model")

Saved XAI model


Listing convolutional layers inside the base model

In [18]:
for layer in model.layers[0].layers:
    if "conv" in layer.name.lower():
        print(layer.name)

conv
conv_bn
expanded_conv_depthwise_pad
expanded_conv_depthwise
expanded_conv_depthwise_bn
expanded_conv_squeeze_excite_avg_pool
expanded_conv_squeeze_excite_conv
expanded_conv_squeeze_excite_relu
expanded_conv_squeeze_excite_conv_1
expanded_conv_squeeze_excite_mul
expanded_conv_project
expanded_conv_project_bn
expanded_conv_1_expand
expanded_conv_1_expand_bn
expanded_conv_1_depthwise_pad
expanded_conv_1_depthwise
expanded_conv_1_depthwise_bn
expanded_conv_1_project
expanded_conv_1_project_bn
expanded_conv_2_expand
expanded_conv_2_expand_bn
expanded_conv_2_depthwise
expanded_conv_2_depthwise_bn
expanded_conv_2_project
expanded_conv_2_project_bn
expanded_conv_2_add
expanded_conv_3_expand
expanded_conv_3_expand_bn
expanded_conv_3_depthwise_pad
expanded_conv_3_depthwise
expanded_conv_3_depthwise_bn
expanded_conv_3_squeeze_excite_avg_pool
expanded_conv_3_squeeze_excite_conv
expanded_conv_3_squeeze_excite_relu
expanded_conv_3_squeeze_excite_conv_1
expanded_conv_3_squeeze_excite_mul
expande

Setting up Google Drive folder structure for the Wafer Pipeline project

In [19]:
BASE_DIR = "/content/drive/MyDrive/Wafer_Pipeline"
MODEL_PATH = BASE_DIR + "/wafer_xai3_model.keras"
CSV_PATH = BASE_DIR + "/Results/predictions.csv"
SELF_LEARN_DIR = "/content/drive/MyDrive/Datasets/Self-learning"
os.makedirs(SELF_LEARN_DIR, exist_ok=True)
OFFLINE_SYNC_DIR = BASE_DIR + "/Offline_Storage/Pending_Sync"
os.makedirs(OFFLINE_SYNC_DIR, exist_ok=True)
SYNCED_DIR = BASE_DIR + "/Offline_Storage/Synced"
os.makedirs(SYNCED_DIR, exist_ok=True)
folders = [
    BASE_DIR,
    BASE_DIR + "/Input_Images",
    BASE_DIR + "/Results",
    BASE_DIR + "/Datasets/Self-learning",
    BASE_DIR + "/Offline_Storage/Pending_Sync"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folder structure created")

Folder structure created


Preprocessing images for model prediction

In [24]:
# ===== IMAGE PREPROCESSING =====
IMG_SIZE = 224
def preprocess_image(img_path):
  img = tf.keras.utils.load_img(
        img_path,
        color_mode="grayscale",
        target_size=(IMG_SIZE, IMG_SIZE)
    )
  img = tf.keras.utils.img_to_array(img)
  img = img / 255.0 # Normalize
  img = tf.repeat(img, 3, axis=-1) # Gray → RGB
  img = tf.expand_dims(img, axis=0) # Add batch dimension
  return img

Function for single-image prediction

In [29]:
def predict_image(img_path):
    img = preprocess_image(img_path)
    preds = model.predict(img, verbose=0)[0]
    class_id = np.argmax(preds)
    confidence = float(np.max(preds))
    label = class_names[class_id]
    return label, confidence

Run prediction on a sample test image and display result

In [30]:
test_image = BASE_DIR + "/Input_Images/open1.png"
label, conf = predict_image(test_image)
print("Prediction :", label)
print("Confidence :", round(conf*100, 2), "%")

Prediction : clean
Confidence : 50.22 %


Initialize results logging CSV file

In [31]:
def init_csv():
    if not os.path.exists(CSV_PATH):
        with open(CSV_PATH, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "Timestamp",
                "Image_Name",
                "Prediction",
                "Confidence",
                "Status"
            ])
        print("CSV file created")
init_csv()

CSV file created


User-facing wrapper to run inspection and display results

In [37]:
CONF_THRESHOLD=0.3
def inspect_image(img_path):
    img_name = os.path.basename(img_path)
    img = preprocess_image(img_path)
    preds = model.predict(img, verbose=0)[0]
    class_id = np.argmax(preds)
    confidence = float(np.max(preds))
    label = class_names[class_id]
    # ---- CHECK CONFIDENCE ----
    if confidence < CONF_THRESHOLD:
        status = "LOW_CONFIDENCE"
        shutil.copy(
            img_path,
            os.path.join(SELF_LEARN_DIR, img_name)
        )
        print("⚠ Sent to Self-learning folder")
    else:
        status = "OK"
    # ---- LOG TO CSV (OFFLINE HISTORY) ----
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(CSV_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            timestamp,
            img_name,
            label,
            round(confidence,4),
            status
        ])
    return label, confidence, status

Run full inspection on a sample image

In [38]:
test_image = BASE_DIR + "/Input_Images/open1.png"
inspect_image(test_image)

('clean', 0.5022304654121399, 'OK')

View inspection log stored in CSV

In [39]:
pd.read_csv(CSV_PATH)

,Timestamp,Image_Name,Prediction,Confidence,Status
0,2026-02-06 18:07:01,open1.png,clean,0.5022,OK


Batch inspection of all images + automatic quality report

In [40]:
def run_batch_inspection():
    files = os.listdir(INPUT_DIR)
    if len(files) == 0:
        print("No images found in Input_Images folder")
        return
    print("Starting Batch Inspection...")
    print("Total Images:", len(files))
    print("-" * 40)
    defect_count = {}
    low_conf_count = 0
    for file in files:
        img_path = os.path.join(INPUT_DIR, file)
        # Skip non-image files
        if not file.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        label, conf, status = inspect_image(img_path)
        # Count defects
        defect_count[label] = defect_count.get(label, 0) + 1
        if status == "LOW_CONFIDENCE":
            low_conf_count += 1
        print(f"{file:20s} → {label:8s} ({conf*100:.1f}%) {status}")
    # ---- SUMMARY ----
    print("\nBatch Summary")
    print("-" * 40)
    total = sum(defect_count.values())
    for k, v in defect_count.items():
        print(f"{k:8s} : {v}")
    print("\nLow Confidence :", low_conf_count)
    yield_percent = (defect_count.get("clean", 0) / total) * 100
    print("Yield :", round(yield_percent, 2), "%")
    if yield_percent < 70:
        print("Batch Status : FAIL")
    else:
        print("Batch Status : PASS")

Sync offline images from Pending to Synced folder

In [41]:
def sync_data():
    pending_files = os.listdir(OFFLINE_SYNC_DIR)
    if len(pending_files) == 0:
        print("Nothing to sync")
        return
    print("Starting Sync...")
    print("-" * 30)
    synced_count = 0
    for file in pending_files:
        src = os.path.join(OFFLINE_SYNC_DIR, file)
        dst = os.path.join(SYNCED_DIR, file)
        # Avoid overwriting
        if os.path.exists(dst):
            continue
        shutil.move(src, dst)
        synced_count += 1
        print(f"Synced: {file}")
    print("-" * 30)
    print("Sync Complete")
    print("Files synced:", synced_count)

Run automated batch inspection on all input images

In [42]:
INPUT_DIR = BASE_DIR + "/Input_Images"
run_batch_inspection()
sync_data()

Starting Batch Inspection...
Total Images: 26
----------------------------------------
crack1.png           → crack    (78.7%) OK
open1.png            → clean    (50.2%) OK
cmp1.png             → cmp      (60.6%) OK
ler1.png             → ler      (38.3%) OK
cmp2.png             → cmp      (55.0%) OK
cmp3.png             → cmp      (53.8%) OK
cmp4.png             → cmp      (32.6%) OK
crack2.png           → crack    (44.9%) OK
crack3.png           → crack    (63.1%) OK
open2.png            → ler      (31.2%) OK
open3.png            → bridge   (47.1%) OK
bridge1.png          → bridge   (70.5%) OK
bridge2.png          → bridge   (50.6%) OK
bridge3.png          → bridge   (52.5%) OK
bridge4.png          → bridge   (47.1%) OK
ler2.png             → ler      (41.6%) OK
⚠ Sent to Self-learning folder
ler3.png             → ler      (27.4%) LOW_CONFIDENCE
⚠ Sent to Self-learning folder
ler4.png             → bridge   (29.1%) LOW_CONFIDENCE
ler5.png             → ler      (44.6%) OK
vias1.png 

In [43]:
import os

MODEL_PATH = "/content/drive/MyDrive/wafer_xai3_model.keras"

size_mb = os.path.getsize(MODEL_PATH) / (1024*1024)
print(f"Model size: {round(size_mb, 2)} MB")


Model size: 5.01 MB
